# Discrete-aware SMOTE-Tomek + Optuna + ensemble learning for diabetes risk prediction
Companion notebook for the manuscript submitted to *Jurnal Ilmiah Teknologi dan Rekayasa* (JITR).
Dataset: **CDC Diabetes Health Indicators (BRFSS 2015)**, file `diabetes_binary_health_indicators_BRFSS2015.csv` (Kaggle: `alexteboul/diabetes-health-indicators-dataset`).

Seven configurations per model (XGBoost, LightGBM, CatBoost, Random Forest):

| Code | Configuration |
|---|---|
| A | Baseline (default hyperparameters) |
| B | Standard SMOTE-Tomek (ST) |
| C | Optuna only |
| D | ST + Optuna |
| E | Optuna + class weighting (CW) |
| F | **Discrete-aware SMOTE-Tomek (ST-D) + Optuna (proposed)** |
| G | ST-D with default hyperparameters (ablation) |

Leakage-free protocol: deduplication → stratified 80:20 split → scaler fitted on training data only → resampling **only** on training data (including inside every Optuna CV fold) → test set used once.

Full run takes about 1 hour on a Colab CPU. For a quick test, set `QUICK = True`.

In [ ]:
!pip -q install optuna catboost imbalanced-learn xgboost lightgbm kagglehub

In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, optuna
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, brier_score_loss, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay)
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
warnings.filterwarnings("ignore"); optuna.logging.set_verbosity(optuna.logging.WARNING)

QUICK = False                      # True = quick test (data subset, few trials)
SEED, NJ = 42, -1                  # NJ = -1: use all cores
N_TUNE, N_FOLDS, N_TRIALS, TIMEOUT = (5000, 3, 5, 60) if QUICK else (30000, 3, 25, 480)
MODELS = ["XGBoost", "LightGBM", "CatBoost", "RandomForest"]
CONFIGS = ["A", "B", "G", "C", "D", "E", "F"]
LABEL = {"A": "Baseline", "B": "ST", "C": "Optuna", "D": "ST+Optuna", "E": "CW+Optuna", "F": "ST-D+Optuna", "G": "ST-D"}

## 1. Load the dataset

In [ ]:
FILE = "diabetes_binary_health_indicators_BRFSS2015.csv"
def load_data():
    if os.path.exists(FILE):                                   # file uploaded manually
        return pd.read_csv(FILE)
    try:                                                       # download from Kaggle
        import kagglehub
        path = kagglehub.dataset_download("alexteboul/diabetes-health-indicators-dataset")
        return pd.read_csv(os.path.join(path, FILE))
    except Exception as e:
        print("kagglehub failed:", e, "-> using a GitHub mirror")
        return pd.read_csv("https://raw.githubusercontent.com/kli252/cdc_diabetes_indicator_dataset/main/" + FILE)
raw = load_data()
print(raw.shape)   # expected (253680, 22)
raw.head()

## 2. Brief EDA

In [ ]:
print("Missing values :", int(raw.isna().sum().sum()))
print("Duplicate rows :", int(raw.duplicated().sum()))
print(raw["Diabetes_binary"].value_counts().rename({0.0: "Non-diabetic", 1.0: "Diabetic"}))
print("Unique values per feature:"); print(raw.drop(columns="Diabetes_binary").nunique().sort_values().to_string())

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
raw["Diabetes_binary"].value_counts().plot.bar(ax=ax[0], color=["#2a78d6", "#eb6834"])
ax[0].set_xticklabels(["Non-diabetic", "Diabetic"], rotation=0); ax[0].set_title("Class distribution")
raw.corr()["Diabetes_binary"].drop("Diabetes_binary").sort_values().plot.barh(ax=ax[1], color="#2a78d6")
ax[1].set_title("Feature correlation with target"); plt.tight_layout(); plt.show()

## 3. Preprocessing: deduplication, stratified split, standardization

In [ ]:
df = raw.drop_duplicates().reset_index(drop=True)
if QUICK: df = df.sample(30000, random_state=SEED).reset_index(drop=True)
X, y = df.drop(columns="Diabetes_binary"), df["Diabetes_binary"].astype(int)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
sc = StandardScaler().fit(Xtr)                                   # fit on training data only
Xtr_s = pd.DataFrame(sc.transform(Xtr), columns=X.columns)
Xte_s = pd.DataFrame(sc.transform(Xte), columns=X.columns)
ytr, yte = ytr.reset_index(drop=True), yte.reset_index(drop=True)
LO, HI = Xtr.min().values, Xtr.max().values
spw = float((ytr == 0).sum() / (ytr == 1).sum())
print(f"After deduplication: {len(df):,} | train {len(Xtr):,} (positive {ytr.sum():,}) | test {len(Xte):,} (positive {yte.sum():,}) | ratio {spw:.2f}:1")

## 4. Resampling: standard vs discrete-aware SMOTE-Tomek
SMOTE interpolates continuously, so binary/ordinal features receive fractional values (e.g. `HighBP = 0.37`) that never occur in real data, and tree models can learn "fractional value = synthetic". The **discrete-aware variant (ST-D)**: SMOTE → back-transform to the original scale → round & clip to the valid range → re-standardize → Tomek links (both members removed, as in standard SMOTE-Tomek).

In [ ]:
def smote_tomek_standard(Xs_, y_):
    return SMOTETomek(random_state=SEED, n_jobs=NJ).fit_resample(Xs_, y_)

def smote_tomek_discrete(Xs_, y_):
    Xo, yo = SMOTE(random_state=SEED).fit_resample(Xs_, y_)
    raw_ = np.clip(np.round(sc.inverse_transform(Xo)), LO, HI)      # snap to valid discrete values
    Xo = pd.DataFrame(sc.transform(raw_), columns=Xs_.columns)
    return TomekLinks(sampling_strategy="all", n_jobs=NJ).fit_resample(Xo, yo)   # same rule as standard SMOTETomek

t = time.time(); Xst, yst = smote_tomek_standard(Xtr_s, ytr);  print(f"ST   : {len(Xst):,} rows {np.bincount(yst)}  ({time.time()-t:.0f} s)")
t = time.time(); Xsd, ysd = smote_tomek_discrete(Xtr_s, ytr);  print(f"ST-D : {len(Xsd):,} rows {np.bincount(ysd)}  ({time.time()-t:.0f} s)")

hb = list(X.columns).index("HighBP")
frac = lambda A: int((np.abs(sc.inverse_transform(A)[:, hb] % 1) > 1e-6).sum())
print("Rows with fractional HighBP -> ST:", frac(Xst), "| ST-D:", frac(Xsd))

## 5. Models and hyperparameter search spaces

In [ ]:
def make(name, p=None, weighted=False):
    p = dict(p or {})
    if name == "XGBoost":
        return XGBClassifier(tree_method="hist", n_jobs=NJ, random_state=SEED, eval_metric="logloss",
                             scale_pos_weight=spw if weighted else 1.0, **p)
    if name == "LightGBM":
        return LGBMClassifier(n_jobs=NJ, random_state=SEED, verbose=-1, class_weight="balanced" if weighted else None, **p)
    if name == "CatBoost":
        return CatBoostClassifier(thread_count=NJ, random_seed=SEED, verbose=0, auto_class_weights="Balanced" if weighted else None, **p)
    return RandomForestClassifier(n_jobs=NJ, random_state=SEED, class_weight="balanced" if weighted else None, **p)

def space(name, tr):
    if name == "XGBoost":
        return dict(n_estimators=tr.suggest_int("n_estimators", 100, 600, step=50), max_depth=tr.suggest_int("max_depth", 3, 10),
                    learning_rate=tr.suggest_float("learning_rate", 0.01, 0.3, log=True), subsample=tr.suggest_float("subsample", 0.6, 1.0),
                    colsample_bytree=tr.suggest_float("colsample_bytree", 0.6, 1.0), min_child_weight=tr.suggest_int("min_child_weight", 1, 10),
                    gamma=tr.suggest_float("gamma", 0.0, 5.0), reg_lambda=tr.suggest_float("reg_lambda", 1e-3, 10.0, log=True))
    if name == "LightGBM":
        return dict(n_estimators=tr.suggest_int("n_estimators", 100, 600, step=50), num_leaves=tr.suggest_int("num_leaves", 15, 255),
                    learning_rate=tr.suggest_float("learning_rate", 0.01, 0.3, log=True), min_child_samples=tr.suggest_int("min_child_samples", 5, 100),
                    subsample=tr.suggest_float("subsample", 0.6, 1.0), subsample_freq=1,
                    colsample_bytree=tr.suggest_float("colsample_bytree", 0.6, 1.0), reg_lambda=tr.suggest_float("reg_lambda", 1e-3, 10.0, log=True))
    if name == "CatBoost":
        return dict(iterations=tr.suggest_int("iterations", 200, 600, step=50), depth=tr.suggest_int("depth", 4, 8),
                    learning_rate=tr.suggest_float("learning_rate", 0.01, 0.3, log=True),
                    l2_leaf_reg=tr.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True), bagging_temperature=tr.suggest_float("bagging_temperature", 0.0, 1.0))
    return dict(n_estimators=tr.suggest_int("n_estimators", 100, 300, step=50), max_depth=tr.suggest_int("max_depth", 6, 30),
                min_samples_split=tr.suggest_int("min_samples_split", 2, 20), min_samples_leaf=tr.suggest_int("min_samples_leaf", 1, 10),
                max_features=tr.suggest_categorical("max_features", ["sqrt", "log2", 0.5]))

## 6. Optuna inside cross-validation (resampling only on the training part of each fold)

In [ ]:
Xsub, _, ysub, _ = train_test_split(Xtr_s, ytr, train_size=N_TUNE, stratify=ytr, random_state=SEED)
Xsub, ysub = Xsub.reset_index(drop=True), ysub.reset_index(drop=True)
folds = []
for tr, va in StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(Xsub, ysub):
    a, b = Xsub.iloc[tr].reset_index(drop=True), ysub.iloc[tr].reset_index(drop=True)
    Xr, yr = smote_tomek_standard(a, b); Xd, yd = smote_tomek_discrete(a, b)
    folds.append(dict(Xtr=a, ytr=b, Xr=Xr, yr=yr, Xd=Xd, yd=yd, Xva=Xsub.iloc[va], yva=ysub.iloc[va]))

def tune(name, mode):   # C = original data, D = ST, E = class weights, F = ST-D
    def objective(trial):
        p, scores = space(name, trial), []
        for f in folds:
            Xa, ya = {"D": (f["Xr"], f["yr"]), "F": (f["Xd"], f["yd"])}.get(mode, (f["Xtr"], f["ytr"]))
            m = make(name, p, weighted=(mode == "E")).fit(Xa, ya)
            scores.append(f1_score(f["yva"], m.predict(f["Xva"])))
        return float(np.mean(scores))
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT)
    return study.best_params, study.best_value

## 7. Experiments: 4 models × 7 configurations

In [ ]:
def evaluate(m):
    pr = m.predict_proba(Xte_s)[:, 1]; yp = (pr >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(yte, yp).ravel()
    return dict(accuracy=accuracy_score(yte, yp), precision=precision_score(yte, yp), recall=recall_score(yte, yp),
                specificity=tn / (tn + fp), f1=f1_score(yte, yp), roc_auc=roc_auc_score(yte, pr),
                pr_auc=average_precision_score(yte, pr), brier=brier_score_loss(yte, pr)), pr

rows, params, probs, models = [], {}, {}, {}
for name in MODELS:
    for cfg in CONFIGS:
        t = time.time(); p, cvf = ({}, None) if cfg in "ABG" else tune(name, cfg)
        Xa, ya = (Xst, yst) if cfg in "BD" else (Xsd, ysd) if cfg in "FG" else (Xtr_s, ytr)
        m = make(name, p, weighted=(cfg == "E")).fit(Xa, ya)
        met, pr = evaluate(m); key = f"{name}_{cfg}"
        rows.append(dict(model=name, config=cfg, cv_f1=cvf, **met)); params[key], probs[key], models[key] = p, pr, m
        print(f"{name:12s} {cfg} {LABEL[cfg]:12s} F1={met['f1']:.4f} Recall={met['recall']:.4f} AUC={met['roc_auc']:.4f} ({time.time()-t:.0f} s)")
res = pd.DataFrame(rows)
res.round(4).to_csv("results.csv", index=False); json.dump(params, open("best_params.json", "w"), indent=1)
res.pivot(index="model", columns="config", values="f1").round(4)

In [ ]:
cols = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "brier"]
res.assign(konfigurasi=res.config.map(LABEL)).set_index(["model", "config", "konfigurasi"])[cols].round(4)

## 8. Best proposed model: classification report & confusion matrix

In [ ]:
best = res[res.config == "F"].sort_values("f1", ascending=False).iloc[0]
BK = f"{best.model}_F"; bm = models[BK]; print("Best proposed model:", BK)
yp = (probs[BK] >= 0.5).astype(int)
print(classification_report(yte, yp, target_names=["Non-diabetic", "Diabetic"], digits=4))
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, c in zip(ax, ["A", "F"]):
    ConfusionMatrixDisplay.from_predictions(yte, (probs[f"{best.model}_{c}"] >= 0.5).astype(int), ax=a, colorbar=False,
                                            display_labels=["Non-diabetic", "Diabetic"], cmap="Blues")
    a.set_title(f"{best.model} – {LABEL[c]}")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for c in ["A", "D", "E", "F"]:
    k = f"{best.model}_{c}"
    RocCurveDisplay.from_predictions(yte, probs[k], ax=ax[0], name=LABEL[c])
    PrecisionRecallDisplay.from_predictions(yte, probs[k], ax=ax[1], name=LABEL[c])
ax[0].plot([0, 1], [0, 1], "k--", lw=0.8); ax[1].axhline(yte.mean(), color="k", ls="--", lw=0.8)
plt.tight_layout(); plt.show()

## 9. 95% bootstrap confidence intervals (F1 & ROC-AUC)

In [ ]:
rng = np.random.default_rng(SEED); B = 200 if QUICK else 1000; yv = yte.values
idx = [rng.integers(0, len(yv), len(yv)) for _ in range(B)]
ci = []
for k, pr in probs.items():
    f1s = [f1_score(yv[i], pr[i] >= 0.5) for i in idx]; aucs = [roc_auc_score(yv[i], pr[i]) for i in idx]
    ci.append(dict(key=k, f1=f"{np.percentile(f1s,2.5):.4f}–{np.percentile(f1s,97.5):.4f}",
                   roc_auc=f"{np.percentile(aucs,2.5):.4f}–{np.percentile(aucs,97.5):.4f}"))
pd.DataFrame(ci)

## 10. Feature importance (built-in & permutation)

In [ ]:
imp = bm.get_feature_importance() if best.model == "CatBoost" else bm.feature_importances_
sub = rng.choice(len(Xte_s), min(20000, len(Xte_s)), replace=False)
pi = permutation_importance(bm, Xte_s.iloc[sub], yte.iloc[sub], scoring="roc_auc", n_repeats=5, random_state=SEED)
fi = pd.DataFrame(dict(feature=X.columns, builtin=np.asarray(imp) / np.sum(imp),
                       perm_mean=pi.importances_mean, perm_std=pi.importances_std)).sort_values("perm_mean")
fi.to_csv("importance.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].barh(fi.feature, fi.builtin, color="#86b6ef"); ax[0].set_title("Built-in importance (normalized)")
ax[1].barh(fi.feature, fi.perm_mean, xerr=fi.perm_std, color="#2a78d6"); ax[1].set_title("Permutation importance (Δ ROC-AUC)")
plt.tight_layout(); plt.show()
fi.sort_values("perm_mean", ascending=False).head(10)